In [11]:
import pandas as pd
import numpy as np

# To load the raw dataset 
df = pd.read_csv('Dataset.csv')
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [12]:
# 1. Flag discount-driven customers
df['Promo_Hunter'] = np.where((df['Discount Applied'] == 'Yes') & (df['Promo Code Used'] == 'Yes'), 1, 0)

# 2. Calculate Estimated Lifetime Value (LTV)
df['Estimated_LTV'] = df['Purchase Amount (USD)'] * (df['Previous Purchases'] + 1)

# 3. Segment customers by value tiers
df['Value_Tier'] = pd.qcut(df['Estimated_LTV'], q=3, labels=['Low Value', 'Mid Value', 'High Value'])

# Preview engineered features
df[['Customer ID', 'Promo_Hunter', 'Estimated_LTV', 'Value_Tier']].head()

,Customer ID,Promo_Hunter,Estimated_LTV,Value_Tier
0,1,1,795,Low Value
1,2,1,192,Low Value
2,3,1,1752,Mid Value
3,4,1,4500,High Value
4,5,1,1568,Mid Value


In [13]:
# Export cleaned data for SQL analysis
df.to_csv('Engineered_D2C_Dataset.csv', index=False)
print("Export successful!")

Export successful!


In [14]:
# Definition 1: Behavioral Loyalty (High frequency + Subscription)
frequent_groups = ['Weekly', 'Bi-Weekly', 'Fortnightly', 'Monthly']
df['Loyalty_Def_1'] = np.where(
    (df['Previous Purchases'] >= 20) & 
    (df['Frequency of Purchases'].isin(frequent_groups)) & 
    (df['Subscription Status'] == 'Yes'), 1, 0
)

# Definition 2: Organic Value Loyalty (High LTV + No Discount Dependency)
median_ltv = df['Estimated_LTV'].median()
df['Loyalty_Def_2'] = np.where(
    (df['Estimated_LTV'] > median_ltv) & (df['Promo_Hunter'] == 0), 1, 0
)

# Set Definition 1 as our primary loyalty label
df['Final_Loyalty_Label'] = df['Loyalty_Def_1']

# Overwrite the final CSV with these new loyalty columns
df.to_csv('Engineered_D2C_Dataset_Final.csv', index=False)
print("Loyalty features successfully engineered and saved!")

Loyalty features successfully engineered and saved!


In [15]:
# Create Satisfaction_Flag based on Review Rating
# Assuming ratings are out of 5. Rating >= 4 is 'Satisfied', else 'Needs Improvement'
df['Satisfaction_Flag'] = np.where(df['Review Rating'] >= 4.0, 'Satisfied', 'Needs Improvement')

# Export the FINAL complete dataset
df.to_csv('Engineered_D2C_Dataset_Final.csv', index=False)
print("Final dataset with all required features is ready!")

Final dataset with all required features is ready!


In [16]:
import sqlite3
import pandas as pd

# 1. Create a local SQLite database connection
conn = sqlite3.connect('d2c_database.db')

# 2. Load the final dataset
df_final = pd.read_csv('Engineered_D2C_Dataset_Final.csv')

# 3. Push data to SQL table named 'customers'
df_final.to_sql('customers', conn, if_exists='replace', index=False)
print("Data successfully loaded into SQL database!")

Data successfully loaded into SQL database!


In [17]:
# Query to count customers in each Value Tier
query_1 = """
SELECT 
    Value_Tier, 
    COUNT("Customer ID") AS Total_Customers,
    SUM("Purchase Amount (USD)") AS Total_Revenue
FROM customers
GROUP BY Value_Tier
ORDER BY Total_Revenue DESC;
"""
result_1 = pd.read_sql_query(query_1, conn)
print("--- Customer Value Distribution ---")
print(result_1)

--- Customer Value Distribution ---
   Value_Tier  Total_Customers  Total_Revenue
0  High Value             1292          98962
1   Mid Value             1308          71552
2   Low Value             1300          62567


In [ ]:
# Query 2: Seasonality, Customer Tenure, and Promo Dependency
query_seasonal = """
SELECT
    o."Season",
    o."Category",
    ROUND(AVG(o."Previous Purchases"), 1)                                    AS avg_prev_purchases,
    COUNT(DISTINCT o."Customer ID")                                          AS unique_customers,
    SUM(CASE WHEN o."Previous Purchases" <= 10 THEN 1 ELSE 0 END)           AS low_tenure_customers,
    SUM(CASE WHEN o."Previous Purchases" >= 35 THEN 1 ELSE 0 END)           AS high_tenure_customers,
    ROUND(AVG(o."Purchase Amount (USD)"), 2)                                 AS avg_spend,
    ROUND(
        SUM(CASE WHEN o."Promo Code Used" = 'Yes' THEN 1 ELSE 0 END)
        * 100.0 / COUNT(*), 1
    )                                                                        AS promo_pct
FROM customers o
GROUP BY o."Season", o."Category"
ORDER BY avg_prev_purchases DESC;
"""
result_seasonal = pd.read_sql_query(query_seasonal, conn)
print("--- Advanced Seasonal & Category Performance ---")
print(result_seasonal)

--- Advanced Seasonal & Category Performance ---
    Season     Category  avg_prev_purchases  unique_customers  \
0   Summer  Accessories                26.8               312   
1   Winter     Footwear                26.7               140   
2   Winter     Clothing                26.1               448   
3   Spring    Outerwear                26.0                81   
4   Spring  Accessories                25.9               301   
5   Summer     Footwear                25.6               160   
6     Fall     Clothing                25.5               427   
7   Winter  Accessories                25.5               303   
8   Winter    Outerwear                25.2                80   
9   Spring     Clothing                24.8               454   
10    Fall  Accessories                24.7               324   
11    Fall     Footwear                24.6               136   
12  Summer     Clothing                24.5               408   
13    Fall    Outerwear                24

In [ ]:
# Query 3: Identifying regions with organic demand vs discount dependency
query_geo = """
SELECT 
    Location,
    COUNT("Customer ID") AS Total_Customers,
    SUM("Purchase Amount (USD)") AS Total_Revenue,
    ROUND(AVG("Purchase Amount (USD)"), 2) AS Avg_Ticket_Size,
    SUM(Promo_Hunter) AS Total_Promo_Hunters,
    ROUND((SUM(Promo_Hunter) * 100.0 / COUNT(*)), 1) AS Promo_Dependency_Pct
FROM customers
GROUP BY Location
ORDER BY Total_Revenue DESC
LIMIT 10;
"""
result_geo = pd.read_sql_query(query_geo, conn)
print("\n--- Geographic Insights & Promo Dependency ---")
print(result_geo)


--- Geographic Insights & Promo Dependency ---
        Location  Total_Customers  Total_Revenue  Avg_Ticket_Size  \
0        Montana               96           5784            60.25   
1       Illinois               92           5617            61.05   
2     California               95           5605            59.00   
3          Idaho               93           5587            60.08   
4         Nevada               87           5514            63.38   
5        Alabama               89           5261            59.11   
6       New York               87           5257            60.43   
7   North Dakota               83           5220            62.89   
8  West Virginia               81           5174            63.88   
9       Nebraska               87           5172            59.45   

   Total_Promo_Hunters  Promo_Dependency_Pct  
0                   36                  37.5  
1                   37                  40.2  
2                   40                  42.1  
3   

In [ ]:
# Query 4: Demographics and preferences of the most loyal customers
query_icp = """
SELECT 
    Gender,
    AVG(Age) AS Avg_Age,
    "Payment Method",
    "Shipping Type",
    COUNT("Customer ID") AS Total_Count
FROM customers
WHERE Final_Loyalty_Label = 1
GROUP BY Gender, "Payment Method", "Shipping Type"
ORDER BY Total_Count DESC
LIMIT 1;
"""
result_icp = pd.read_sql_query(query_icp, conn)
print("\n--- Ideal Customer Profile (ICP) Basis Loyalty ---")
print(result_icp)


--- Ideal Customer Profile (ICP) Basis Loyalty ---
  Gender    Avg_Age Payment Method  Shipping Type  Total_Count
0   Male  45.952381          Venmo  Free Shipping           21
